In [13]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [14]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

model = init_chat_model(
    model_provider="openai",
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model=os.environ.get("OPENAI_MODEL"),
    temperature=0.3,
)


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [19]:
from langchain.messages import HumanMessage, AIMessage
from rich import print as rprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is engaging in a
role-play scenario set in Lunapolis, the capital city of the moon. The primary goal is to navigate a political 
crisis where the new President (the AI) must address the dissatisfaction of the 100,000-strong Cheese Miners' 
Union, who are threatening to strike due to unsafe working conditions and unfair quotas.\n\n## SUMMARY\n- 
**World-Building Context**:\n  - **Location**: Lunapolis (Moon's capital).\n  - **Environment**: Extreme thermal 
fluctuations (High: 120°C, Low: -100°C) with clear skies.\n  - **Demographics**: 100,000 cheese miners form a 
significant portion of the population/workforce.\n- **Conflict Dynamics**: The Union is unhappy with the new 
administration; a strike is deemed likely if not addressed.\n- **Strategic Response Formulated**: The AI, acting as
President, delivered a speech outlining a three-point plan to avert the strike:\n  1.  **Immediate Moratorium**: 
Suspended increased production quotas to prioritize safety.\n  2.  **Joint Tribunal**: Established a body with 
Union reps and independent engineers to review wages and audit safety equipment within 48 hours.\n  3.  
**Transparency**: Pledged to open budget books to Union auditors to ensure profit reinvestment into habitat 
maintenance.\n- **Current Status**: The President has invited the Union leadership to negotiate at the Presidential
Dome. The specific arguments regarding safety suits, life-support systems, and hazard pay have been established as 
key negotiation points.\n\n## ARTIFACTS\nNone. No external files were created or modified. The conversation 
consists entirely of hypothetical world-building and role-play dialogue.\n\n## NEXT STEPS\nSimulate the response of
the Cheese Miners' Union leadership to the President's three-point plan and the invitation to negotiate. Determine 
if the strike is averted or if further concessions are required based on the Union's reaction to the proposed 
Tribunal and transparency measures.",
            additional_kwargs={'lc_source': 'summarization'},
            response_metadata={},
            id='85cc9961-c415-4208-bca9-1a8b6715ba01'
        ),
        HumanMessage(
            content="If you were Lunapolis' new president how would you respond to the cheese miners' union?",
            additional_kwargs={},
            response_metadata={},
            id='886fa769-9555-4fec-8eb0-2b3e8cb53cee'
        ),
        AIMessage(
            content='Fellow Citizens of Lunapolis, Miners of the Great Regolith, and Leaders of the Cheese Miners\'
Union:\n\nI hear you. The silence of the mines is a louder warning than any siren we have in the Dome. You have 
made it clear: **no amount of cheese is worth a single life lost to the vacuum, the cold, or a failing 
seal.**\n\nAs your new President, I am not here to offer you platitudes while you freeze in the shadows. I am here 
to offer you action. Effective immediately, I am enacting the **"Lunar Safety First" Directive**, comprised of 
three non-negotiable pillars:\n\n### 1. Immediate Moratorium on Quotas\nThe increased production targets set by the
previous administration are **suspended indefinitely**. Until every single mining sector is certified safe by 
*your* chosen engineers, production will run only at a pace that guarantees 100% crew safety. We will eat less 
cheese before we bury another miner.\n\n### 2. The Joint Safety & Wage Tribunal\nWithin the next **48 hours**, I am
establishing a tribunal to audit our operations. This body will not be stacked with my appointees. It will consist 
of:\n*   Three representatives elected directly by the Union.\n*   Three independent structural engineers from the 
Earth-Moon Trade Federation (neutral third parties).\n*   Two members of my administration.\n\n**Their mandate:** 
To inspect every life-support system, thermal suit, and excavation rig. They have the auth

In [20]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is engaging in a role-play scenario set in Lunapolis, the capital city of the moon. The primary goal is to navigate a political crisis where the new President (the AI) must address the dissatisfaction of the 100,000-strong Cheese Miners' Union, who are threatening to strike due to unsafe working conditions and unfair quotas.

## SUMMARY
- **World-Building Context**:
  - **Location**: Lunapolis (Moon's capital).
  - **Environment**: Extreme thermal fluctuations (High: 120°C, Low: -100°C) with clear skies.
  - **Demographics**: 100,000 cheese miners form a significant portion of the population/workforce.
- **Conflict Dynamics**: The Union is unhappy with the new administration; a strike is deemed likely if not addressed.
- **Strategic Response Formulated**: The AI, acting as President, delivered a speech outlining a three-point plan to avert the strike:
  1.  **Immediate Moratorium**: Suspended increased productio

## Trim/delete messages

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [ ]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

In [ ]:
print(response["messages"][-1].content)